# Déformer une grille

| Module | Durée | Prérequis |
|---|---|---|
| M3 – Matrices et transformations | 75 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Maëlle, graphiste, dessine le logo d'une association : une grande lettre F. Son logiciel de dessin propose cinq boutons :

| Bouton | Effet annoncé |
|---|---|
| Pivoter | tourner le logo d'un angle choisi |
| Agrandir | l'agrandir ou le réduire sans le déformer |
| Étirer | l'allonger dans une seule direction |
| Incliner | le pencher, comme une police en italique |
| Aplatir | l'écraser sur une droite, pour dessiner une ombre au sol |

Le logo compte des milliers de points. Pourtant, le logiciel applique chaque effet instantanément, et le fichier enregistre chaque effet avec **quatre nombres seulement**.

**Les questions de Maëlle : quels sont ces quatre nombres ? Comment le logiciel s'en sert-il pour déplacer tous les points ?**

**Activité** : Quelles informations faudrait-il connaître pour décrire complètement un effet ? Pensez-vous qu'il suffise de savoir où vont deux points bien choisis ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Voici une matrice mystère :

$$
M = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}
$$

Avant tout calcul, pariez : quel est son effet sur le logo ?

In [ ]:
pari_effet_M = "symétrie"   # "rotation", "symétrie", "étirement", "inclinaison", "aplatissement" (modifiez)

## ③ Abstraire : une matrice est une action

Au module M2, une matrice était un tableau de coefficients. Changeons de regard : une matrice $A$ de taille $2 \times 2$ **agit** sur les points du plan. Elle envoie le point $\mathbf{x} = (x\,;\,y)$ sur le point $A\mathbf{x}$ :

$$
\begin{pmatrix} a & b \\ c & d \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} = \begin{pmatrix} a\,x + b\,y \\ c\,x + d\,y \end{pmatrix}
$$

Une telle action est dite **linéaire** : elle respecte les combinaisons linéaires du module M1.

$$
A(\lambda\,\mathbf{u} + \mu\,\mathbf{v}) = \lambda\,A\mathbf{u} + \mu\,A\mathbf{v}
$$

**La conséquence clé.** Tout point s'écrit $\mathbf{x} = x\,\mathbf{e}_1 + y\,\mathbf{e}_2$, avec $\mathbf{e}_1 = (1\,;\,0)$ et $\mathbf{e}_2 = (0\,;\,1)$. Donc

$$
A\mathbf{x} = x\,A\mathbf{e}_1 + y\,A\mathbf{e}_2 .
$$

Il suffit de savoir où vont $\mathbf{e}_1$ et $\mathbf{e}_2$ pour savoir où va **tout** le plan. Et ces deux images sont exactement les **colonnes** de la matrice.

![Schéma : le carré unité et la grille avant et après la transformation ; les images de e1 et e2 sont les colonnes de la matrice](../../assets/schemas/schema-colonnes-matrice.svg)

**Activité** : Avec la matrice du schéma, $A = \begin{pmatrix} 2 & 1 \\ 0{,}5 & 1{,}5 \end{pmatrix}$, calculez l'image du point $(1\,;\,2)$ de deux façons : avec la formule du produit, puis comme $1 \times A\mathbf{e}_1 + 2 \times A\mathbf{e}_2$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les outils de dessin

La cellule suivante définit la lettre F, une fonction `dessiner(ax, A, titre)` et le catalogue des transformations. Lisez-la : chaque transformation tient en une ligne.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Un point est une COLONNE : un ensemble de n points est un tableau de forme (2, n).
# Ainsi, appliquer la matrice A à tous les points s'écrit simplement A @ points.

# La lettre F : asymétrique, elle montre les rotations ET les retournements
F = np.array([[0, 0, 2, 2, 0.5, 0.5, 1.5, 1.5, 0.5, 0.5, 0],
              [0, 3, 3, 2.5, 2.5, 1.7, 1.7, 1.2, 1.2, 0, 0]])

def grille(ax, A=np.eye(2), n=4, **style):
    # Trace l'image par A des droites x = k et y = k, pour k entre -n et n
    t = np.linspace(-n, n, 2)
    for k in range(-n, n + 1):
        for ligne in (np.array([t, np.full(2, k)]), np.array([np.full(2, k), t])):
            image = A @ ligne
            ax.plot(*image, lw=0.6, color=style.get("couleur", "#D5DBDE"), zorder=0)

def dessiner(ax, A, titre="", limite=4.5):
    # Grille transformée, lettre F avant (gris) et après (vert), images des vecteurs de base
    grille(ax, A)
    ax.fill(*F, color=LIOUAI["gris"], alpha=0.35, lw=0)
    image = A @ F
    ax.fill(*image, color=LIOUAI["vert"], alpha=0.55)
    ax.plot(*image, color=LIOUAI["vert_texte"], lw=1.5)
    for colonne, couleur in [(A[:, 0], LIOUAI["vert_texte"]), (A[:, 1], LIOUAI["navy"])]:
        ax.annotate("", xy=colonne, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", lw=2, color=couleur, mutation_scale=14))
    ax.set_xlim(-limite, limite); ax.set_ylim(-limite, limite); ax.set_aspect("equal")
    ax.axhline(0, color=LIOUAI["gris"], lw=0.8); ax.axvline(0, color=LIOUAI["gris"], lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(titre, fontsize=11)

# Le catalogue des transformations
def rotation(theta_deg):
    t = np.radians(theta_deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def homothetie(k):
    return k * np.eye(2)

def dilatation(kx, ky):
    return np.array([[kx, 0], [0, ky]])

def cisaillement(k):
    return np.array([[1, k], [0, 1]])

def projection_sur_x():
    return np.array([[1, 0], [0, 0]])

def symetrie_axe_y():
    return np.array([[-1, 0], [0, 1]])

In [ ]:
A = np.array([[2, 1], [0.5, 1.5]])
print("Calcul avec le produit     :", A @ np.array([1, 2]))
print("Combinaison des colonnes   :", 1 * A[:, 0] + 2 * A[:, 1])

### Le catalogue de Maëlle

Les flèches montrent les images de $\mathbf{e}_1$ (en vert) et de $\mathbf{e}_2$ (en bleu marine), c'est-à-dire les colonnes de chaque matrice. Le F gris est le logo de départ, le F vert le résultat.

In [ ]:
catalogue = {
    "Pivoter de 30°": rotation(30),
    "Agrandir × 1,5": homothetie(1.5),
    "Étirer en largeur × 2": dilatation(2, 1),
    "Incliner (italique)": cisaillement(0.4),
    "Aplatir sur l'axe x": projection_sur_x(),
    "Retourner (miroir)": symetrie_axe_y(),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (nom, A) in zip(axes.flat, catalogue.items()):
    dessiner(ax, A, nom)
plt.tight_layout(); plt.show()

for nom, A in catalogue.items():
    print(f"{nom:24s} {np.round(A, 3).tolist()}")

**Activité** : Pour chaque effet, vérifiez sur le dessin que la flèche verte est bien la première colonne, et la flèche marine la seconde. Puis écrivez la matrice de rotation d'angle $\theta$ en lisant où vont $\mathbf{e}_1$ et $\mathbf{e}_2$ :

$$
R_\theta = \begin{pmatrix} \;?\; & \;?\; \\ \;?\; & \;?\; \end{pmatrix}
$$

Indiquez vos réponses et justifiez-les.

### La matrice mystère

In [ ]:
M = np.array([[0, -1], [1, 0]])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
dessiner(ax, M, "La matrice mystère")
plt.show()
print("Image de e1 :", M[:, 0], "   Image de e2 :", M[:, 1])
print("Comparaison avec rotation(90) :", np.allclose(M, rotation(90)))

### Lire une matrice à partir de son effet

**Activité** : Trouvez, sans ordinateur, la matrice qui :

1. envoie $\mathbf{e}_1$ sur $(0\,;\,2)$ et $\mathbf{e}_2$ sur $(-1\,;\,0)$ ;
2. retourne le logo par rapport à l'axe horizontal ;
3. étire les hauteurs par 3 sans changer les largeurs.

Complétez la cellule, puis vérifiez avec `dessiner`.

In [ ]:
A1 = None   # à compléter
A2 = None
A3 = None

### Et pour une vraie image ?

Une photo n'est pas une liste de points, mais une grille de pixels. Pour la transformer, on fait le chemin **inverse** : pour chaque pixel de l'image d'arrivée, on cherche de quel pixel de départ il provient. Il faut donc pouvoir « défaire » la transformation, ce que le module M4 étudiera. La fonction `scipy.ndimage.affine_transform` s'en charge ; ici, nous lui donnons simplement la matrice inverse.

In [ ]:
from scipy import ndimage

# Une « photo » synthétique : un damier et un disque, 200 x 200 pixels
n = 200
yy, xx = np.mgrid[0:n, 0:n]
photo = (((xx // 25) + (yy // 25)) % 2) * 0.35 + 0.15
photo[(xx - 120) ** 2 + (yy - 80) ** 2 < 30 ** 2] = 1.0

def transformer_image(img, A):
    # Applique A autour du centre de l'image (les indices numpy sont dans l'ordre ligne, colonne)
    A_lc = np.array([[A[1, 1], A[1, 0]], [A[0, 1], A[0, 0]]])     # A exprimée en (ligne, colonne)
    inverse = np.linalg.inv(A_lc)
    centre = np.array(img.shape) / 2
    decalage = centre - inverse @ centre
    return ndimage.affine_transform(img, inverse, offset=decalage, order=1, cval=1.0)

effets = {"Départ": np.eye(2), "Pivoter de 30°": rotation(30), "Incliner": cisaillement(0.4), "Étirer × 1,4": dilatation(1.4, 1)}
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, (nom, A) in zip(axes, effets.items()):
    ax.imshow(transformer_image(photo, A), cmap="liouai_r", origin="lower")
    ax.set_title(nom, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## ⑤ Interpréter

Mesurons ce que chaque effet conserve : les longueurs, les angles, les aires et le sens (un F à l'endroit reste-t-il à l'endroit ?).

In [ ]:
def aire(polygone):
    # Formule du lacet pour l'aire d'un polygone fermé (points en colonnes)
    x, y = polygone
    return 0.5 * abs(np.dot(x[:-1], y[1:]) - np.dot(x[1:], y[:-1]))

def sens_conserve(A):
    # Le F reste-t-il à l'endroit ? On regarde si (Ae1, Ae2) tourne dans le même sens que (e1, e2)
    orientation = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    if np.isclose(orientation, 0):
        return "—"                                # figure aplatie : la question n'a plus de sens
    return "oui" if orientation > 0 else "non"

carre = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])
print(f"{'effet':24s} {'longueur de e1':>14s} {'angle(Ae1,Ae2)':>15s} {'aire du carré':>14s} {'sens':>6s}")
for nom, A in catalogue.items():
    c1, c2 = A[:, 0], A[:, 1]
    longueur = np.linalg.norm(c1)
    if np.linalg.norm(c2) > 0:
        angle = np.degrees(np.arccos(np.clip(c1 @ c2 / (np.linalg.norm(c1) * np.linalg.norm(c2)), -1, 1)))
    else:
        angle = None
    texte_angle = f"{angle:14.1f}°" if angle is not None else f"{'—':>15s}"
    print(f"{nom:24s} {longueur:14.2f} {texte_angle} {aire(A @ carre):14.2f} {sens_conserve(A):>6s}")

In [ ]:
print(f"Votre pari pour M : « {pari_effet_M} » ; M est la rotation d'un quart de tour (90°).")

**Activité** :

1. Quels effets conservent les longueurs ? les angles ? les aires ? Classez-les dans un tableau.
2. Le bouton « Aplatir » donne une aire nulle. Peut-on retrouver le logo d'origine à partir de son ombre ? Pourquoi ?
3. Le miroir conserve longueurs, angles et aires, mais pas le sens. Comment le voit-on sur le F ?
4. Notez les aires obtenues : vous les retrouverez au module M4, sous le nom de **déterminant**.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** Toutes les transformations du catalogue laissent l'origine $(0\,;\,0)$ fixe. Maëlle voudrait aussi un bouton « Déplacer ». Peut-on l'écrire avec une matrice $2 \times 2$ ? Gardez la question pour le notebook 3.3.
- **Vérifier autrement.** La linéarité $A(\mathbf{u} + \mathbf{v}) = A\mathbf{u} + A\mathbf{v}$ se teste numériquement sur des vecteurs tirés au hasard. Faites-le pour trois matrices du catalogue.
- **Généraliser.** Écrivez une fonction `matrice_depuis_images(image_e1, image_e2)` qui renvoie la matrice correspondante. Utilisez-la pour vos trois réponses de « Lire une matrice ».

In [ ]:
rng = np.random.default_rng(1)
u, v = rng.normal(size=2), rng.normal(size=2)
for nom in ["Pivoter de 30°", "Incliner (italique)", "Aplatir sur l'axe x"]:
    A = catalogue[nom]
    print(f"{nom:22s} A(u+v) = A u + A v ? {np.allclose(A @ (u + v), A @ u + A @ v)}")

In [ ]:
def matrice_depuis_images(image_e1, image_e2):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def matrice_depuis_images(image_e1, image_e2):
    return np.column_stack([image_e1, image_e2])

A1 = matrice_depuis_images([0, 2], [-1, 0])     # étirement x 2 selon x, puis quart de tour
A2 = matrice_depuis_images([1, 0], [0, -1])     # miroir par rapport à l'axe horizontal
A3 = matrice_depuis_images([1, 0], [0, 3])      # hauteurs x 3
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, A, nom in zip(axes, [A1, A2, A3], ["A1", "A2 : miroir horizontal", "A3 : hauteurs × 3"]):
    dessiner(ax, A, nom, limite=6)
plt.tight_layout(); plt.show()
print("A1 =", A1.tolist(), "= rotation(90) @ dilatation(2, 1) :", np.allclose(A1, rotation(90) @ dilatation(2, 1)))
print("Rotation d'angle θ : colonnes (cos θ ; sin θ) et (-sin θ ; cos θ)")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Décrire le concept": 1,
    "Expliquer visuellement": 1,
    "Relier les traits du résultat à leur sens réel": 1,
    "Estimer la solution avant de calculer": 1,
    "Appliquer l'outil": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "3.1-deformer-une-grille", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")